> **Chapter Map:** This notebook is the companion for **Chapter 6** (Zeolites and Carbon Nanotubes in Catalysis).

# NB6: Zeolites and Carbon Nanotubes in Catalysis

## Learning Objectives

After completing this notebook, you will be able to:

1. Compare zeolite framework types by pore geometry, window size, and cage dimensions
2. Calculate Brønsted acid site density from Si/Al ratio and unit cell composition
3. Model the three types of shape selectivity (reactant, product, transition-state) using molecular size vs. pore size criteria
4. Implement configurational diffusion and compare it with Knudsen and bulk diffusion regimes
5. Calculate the effectiveness factor for microporous zeolite crystals and assess diffusion limitations
6. Analyze para-xylene selectivity in MFI zeolite as a product shape selectivity case study
7. Calculate CNT diameter from chiral indices and classify chirality and electronic character
8. Compare SWCNT and MWCNT properties relevant to catalytic applications
9. Compute classical Knudsen diffusivity and enhanced CNT diffusivity with the enhancement factor $F$
10. Quantify how enhanced transport shifts the effectiveness factor curve, expanding the kinetic-control window
11. Design a CNT-supported catalyst to achieve a target effectiveness factor
12. Compare CNT, conventional, and zeolite supports in terms of transport-limited performance

## Background

Zeolites and carbon nanotubes are both nanoscale materials used as catalysts or catalyst supports,
but they sit at opposite ends of the transport spectrum. **Zeolites** are crystalline
aluminosilicates whose micropore dimensions (3--8 Å) are comparable to molecular dimensions,
enabling **shape selectivity** but imposing severe diffusion limitations through configurational
diffusion ($D \sim 10^{-10}$ to $10^{-18}$ m$^2$/s). **Carbon nanotubes** have atomically smooth
graphitic walls that enable **enhanced transport** -- diffusivities 10--10,000 times faster than
classical Knudsen predictions -- dramatically expanding the kinetic-control window.

This notebook explores both material classes, building intuition for how pore structure controls
transport, selectivity, and catalyst effectiveness.

### Prerequisites

- **Concepts:** Thiele modulus, effectiveness factor (Chapter 4), diffusion regimes (Chapters 4--5)
- **Python skills:** numpy arrays, matplotlib plotting, pandas DataFrames
- **Previous notebooks:** NB4 (Transport Limitations), NB5 (Selectivity Engineering)

---
## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Patch
import pandas as pd

# Set default plot style for publication-quality figures
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['legend.fontsize'] = 11
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11
plt.rcParams['lines.linewidth'] = 2
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Physical constants
R = 8.314  # Gas constant in J/(mol*K)

# Colorblind-safe color palette (Wong, 2011)
COLORS = {
    'blue': '#0072B2',
    'orange': '#E69F00',
    'green': '#009E73',
    'vermillion': '#D55E00',
    'skyblue': '#56B4E9',
    'purple': '#CC79A7'
}

print("Setup complete. Libraries imported successfully.")

---
## Part 1: Zeolite Pore Geometry

Zeolites are classified by their pore size, which is determined by the number of T-atoms
(Si or Al) in the ring that forms the pore opening:

| Category | Ring Size | Pore Diameter | Examples |
|----------|-----------|---------------|----------|
| Small pore | 8-MR | 3--4 angstrom | CHA, LTA |
| Medium pore | 10-MR | 5--6 angstrom | MFI, FER |
| Large pore | 12-MR | 6--8 angstrom | FAU, BEA, MOR |
| Extra-large | >12-MR | >8 angstrom | UTD-1, ITQ-33 |

In [ ]:
# Common zeolite framework data
# Pore dimensions from IZA database and lecture notes
frameworks = pd.DataFrame({
    'Framework': ['FAU', 'BEA', 'MOR', 'MFI', 'FER', 'CHA', 'LTA'],
    'Common_Name': ['Zeolite Y', 'Beta', 'Mordenite', 'ZSM-5',
                    'Ferrierite', 'SSZ-13', 'Zeolite A'],
    'Ring_Size': [12, 12, 12, 10, 10, 8, 8],
    'Pore_Diameter_A': [7.4, 6.7, 7.0, 5.5, 5.4, 3.8, 4.1],
    'Channel_Dim': ['3D', '3D', '1D', '3D', '2D', '3D', '3D'],
    # Largest internal void: supercage (FAU, LTA), channel intersection (MFI),
    # or cage (CHA).  MFI's 9 A void is a channel intersection, not a true cage.
    'Cavity_Diameter_A': [13.0, np.nan, np.nan, 9.0, np.nan, 7.3, 11.4],
    'T_atoms_per_UC': [192, 64, 48, 96, 36, 36, 24],
    'Category': ['Large', 'Large', 'Large', 'Medium',
                 'Medium', 'Small', 'Small']
})

print("Zeolite Framework Comparison")
print("=" * 80)
print(frameworks.to_string(index=False))

In [ ]:
# Bar chart comparing pore sizes across frameworks
fig, ax = plt.subplots(figsize=(10, 6))

# Color by category
category_colors = {
    'Small': COLORS['vermillion'],
    'Medium': COLORS['orange'],
    'Large': COLORS['blue']
}
bar_colors = [category_colors[cat] for cat in frameworks['Category']]

bars = ax.bar(frameworks['Framework'], frameworks['Pore_Diameter_A'],
              color=bar_colors, edgecolor='black', linewidth=1.2, alpha=0.85)

# Add common name labels on bars
for bar, name in zip(bars, frameworks['Common_Name']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.15,
            name, ha='center', va='bottom', fontsize=9, style='italic')

# Add ring size annotations inside bars
for bar, ring in zip(bars, frameworks['Ring_Size']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() / 2,
            f'{ring}-MR', ha='center', va='center', fontsize=10,
            fontweight='bold', color='white')

# Horizontal reference lines for molecular sizes
molecules = {
    'n-hexane (4.3)': 4.3,
    'benzene (5.9)': 5.9,
    'neopentane (6.2)': 6.2
}
line_styles = ['--', '-.', ':']
line_colors = ['gray', 'gray', 'gray']

for (label, size), ls in zip(molecules.items(), line_styles):
    ax.axhline(y=size, color='gray', linestyle=ls, linewidth=1.2, alpha=0.6)
    ax.text(6.6, size + 0.1, label, fontsize=9, color='gray', ha='right')

# Legend for categories
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=COLORS['vermillion'], edgecolor='black', label='Small pore (8-MR)'),
    Patch(facecolor=COLORS['orange'], edgecolor='black', label='Medium pore (10-MR)'),
    Patch(facecolor=COLORS['blue'], edgecolor='black', label='Large pore (12-MR)')
]
ax.legend(handles=legend_elements, loc='upper right')

ax.set_xlabel('Framework Type (IZA Code)')
ax.set_ylabel('Pore Window Diameter (\u00c5)')
ax.set_title('Zeolite Pore Size Comparison')
ax.set_ylim(0, 9)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### Key Observations

1. **Pore size determines molecular access**: n-hexane (4.3 angstrom) can enter all zeolites, but neopentane (6.2 angstrom) cannot enter small or medium pore zeolites
2. **Channel dimensionality matters**: MOR has 1D channels (susceptible to pore blocking), while FAU and MFI have 3D channel systems
3. **Cage vs. channel**: FAU has large supercages (13 angstrom) accessible through 7.4 angstrom windows; once inside, molecules have more room

---
## Part 2: Brønsted Acid Site Density

When Al$^{3+}$ substitutes for Si$^{4+}$ in the zeolite framework, a negative charge is
introduced. A charge-compensating proton creates a **bridging hydroxyl group** (Si--OH--Al)
that acts as a Brønsted acid site.

**Løwenstein's rule** forbids Al--O--Al linkages, so Si/Al $\geq$ 1.

The acid site density calculation:
1. From Si/Al ratio and total T-atoms per unit cell, find the number of Al atoms
2. Compute the unit cell molar mass: $M_{\text{UC}} = n_{\text{Al}} \times 27 + n_{\text{Si}} \times 28 + n_{\text{O}} \times 16$
3. Each Al generates one Brønsted acid site (ideally)

In [ ]:
def acid_site_density(si_al_ratio, T_atoms_per_UC, n_O_per_UC=None):
    """
    Calculate Brønsted acid site density from Si/Al ratio.

    For a zeolite with T_atoms total tetrahedral atoms per unit cell
    and a given Si/Al ratio, this function computes the number of Al
    atoms per unit cell, the unit cell mass, and the theoretical acid
    site density.

    Parameters
    ----------
    si_al_ratio : float
        Molar Si/Al ratio (must be >= 1 by Løwenstein's rule)
    T_atoms_per_UC : int
        Total number of T-atoms (Si + Al) per unit cell
    n_O_per_UC : int, optional
        Number of oxygen atoms per unit cell. If None, estimated
        as 2 * T_atoms_per_UC (each T-atom shares 4 oxygens with neighbors).

    Returns
    -------
    dict
        Dictionary with keys:
        - 'n_Al': Number of Al per unit cell
        - 'n_Si': Number of Si per unit cell
        - 'M_UC': Unit cell molar mass (g/mol)
        - 'acid_density_mmol_g': Acid site density (mmol/g)
        - 'acid_density_mol_g': Acid site density (mol/g)

    Notes
    -----
    Assumes every framework Al generates one Brønsted acid site.
    Real catalysts may have fewer accessible sites due to EFAL,
    inaccessible locations, or framework defects.
    """
    if si_al_ratio < 1:
        raise ValueError("Si/Al must be >= 1 (Løwenstein's rule)")

    # Solve: Si/Al = (T - x) / x  =>  x = T / (1 + Si/Al)
    n_Al = T_atoms_per_UC / (1 + si_al_ratio)
    n_Si = T_atoms_per_UC - n_Al

    if n_O_per_UC is None:
        n_O_per_UC = 2 * T_atoms_per_UC

    # Unit cell molar mass
    M_UC = n_Al * 27.0 + n_Si * 28.0 + n_O_per_UC * 16.0  # g/mol

    # Acid site density: one site per Al atom
    acid_mol_g = n_Al / M_UC  # mol/g
    acid_mmol_g = acid_mol_g * 1000  # mmol/g

    return {
        'n_Al': round(n_Al, 2),
        'n_Si': round(n_Si, 2),
        'M_UC': round(M_UC, 1),
        'acid_density_mmol_g': round(acid_mmol_g, 3),
        'acid_density_mol_g': acid_mol_g
    }

In [ ]:
# Worked example: H-ZSM-5 with Si/Al = 25
# MFI has 96 T-atoms and 192 oxygens per unit cell
result = acid_site_density(si_al_ratio=25, T_atoms_per_UC=96, n_O_per_UC=192)

print("Worked Example: H-ZSM-5 (MFI) with Si/Al = 25")
print("=" * 50)
print(f"Al per unit cell: {result['n_Al']:.1f}")
print(f"Si per unit cell: {result['n_Si']:.1f}")
print(f"Unit cell mass: {result['M_UC']:.0f} g/mol")
print(f"Acid site density: {result['acid_density_mmol_g']:.2f} mmol/g")
print()
print("If NH3-TPD measures 0.35 mmol/g:")
f_accessible = 0.35 / result['acid_density_mmol_g']
print(f"Fraction of Al generating accessible sites: {f_accessible:.0%}")

In [ ]:
# Acid site density vs Si/Al ratio for different frameworks
si_al_range = np.linspace(1, 100, 200)

# Compute acid site density for MFI (96 T-atoms, 192 O)
acid_densities_mfi = []
for ratio in si_al_range:
    res = acid_site_density(ratio, 96, 192)
    acid_densities_mfi.append(res['acid_density_mmol_g'])

# Compute for FAU (192 T-atoms, 384 O)
acid_densities_fau = []
for ratio in si_al_range:
    res = acid_site_density(ratio, 192, 384)
    acid_densities_fau.append(res['acid_density_mmol_g'])

# Compute for CHA (36 T-atoms, 72 O)
acid_densities_cha = []
for ratio in si_al_range:
    res = acid_site_density(ratio, 36, 72)
    acid_densities_cha.append(res['acid_density_mmol_g'])

fig, ax = plt.subplots(figsize=(10, 7))

ax.plot(si_al_range, acid_densities_mfi, color=COLORS['blue'],
        linestyle='-', linewidth=2.5, label='MFI (ZSM-5, 96 T-atoms)')
ax.plot(si_al_range, acid_densities_fau, color=COLORS['orange'],
        linestyle='--', linewidth=2.5, label='FAU (Zeolite Y, 192 T-atoms)')
ax.plot(si_al_range, acid_densities_cha, color=COLORS['green'],
        linestyle='-.', linewidth=2.5, label='CHA (SSZ-13, 36 T-atoms)')

# Mark typical commercial Si/Al ratios
commercial = {
    'MFI': (25, COLORS['blue']),
    'FAU': (5, COLORS['orange']),
    'CHA': (15, COLORS['green'])
}
for label, (ratio, color) in commercial.items():
    if label == 'MFI':
        y_val = acid_site_density(ratio, 96, 192)['acid_density_mmol_g']
    elif label == 'FAU':
        y_val = acid_site_density(ratio, 192, 384)['acid_density_mmol_g']
    else:
        y_val = acid_site_density(ratio, 36, 72)['acid_density_mmol_g']
    ax.plot(ratio, y_val, 'o', color=color, markersize=12,
            markeredgecolor='black', markeredgewidth=1.5)
    ax.annotate(f'{label}\nSi/Al={ratio}', xy=(ratio, y_val),
                xytext=(ratio + 5, y_val + 0.15),
                fontsize=10, color=color,
                arrowprops=dict(arrowstyle='->', color=color))

# Add Løwenstein limit
ax.axvline(x=1, color='gray', linestyle=':', linewidth=1.5, alpha=0.5)
ax.text(1.5, 0.3, "Løwenstein's\nlimit (Si/Al=1)",
        fontsize=9, color='gray')

ax.set_xlabel('Si/Al Molar Ratio')
ax.set_ylabel('Acid Site Density (mmol/g)')
ax.set_title('Brønsted Acid Site Density vs. Si/Al Ratio')
ax.set_xlim(0, 105)
ax.set_ylim(0, None)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Key Observations

1. Acid site density drops rapidly with increasing Si/Al ratio (hyperbolic relationship)
2. All framework types converge at high Si/Al (few acid sites regardless of structure)
3. At low Si/Al, FAU has the highest density because its larger unit cell formula normalizes similarly
4. **Trade-off**: Low Si/Al = many but weaker sites; high Si/Al = few but stronger sites

---
## Part 3: Shape Selectivity Modeling

Shape selectivity arises because zeolite pore dimensions are comparable to molecular dimensions.
We model this by comparing the **critical molecular diameter** to the **pore window diameter**.

| Molecule | Critical Diameter (angstrom) | Shape |
|----------|---------------------|-------|
| H$_2$ | 2.9 | Linear |
| N$_2$ | 3.6 | Linear |
| n-hexane | 4.3 | Linear |
| cyclohexane | 6.0 | Cyclic |
| benzene | 5.9 | Planar |
| p-xylene | 5.8 | Planar (compact) |
| o-xylene | 6.8 | Planar (bulky) |
| m-xylene | 6.8 | Planar (bulky) |
| neopentane | 6.2 | Spherical |
| 1,3,5-TIPB | 8.5 | Bulky aromatic |

In [ ]:
# Molecular diameters and zeolite pore sizes
molecules = pd.DataFrame({
    'Molecule': ['H2', 'N2', 'n-hexane', 'cyclohexane', 'benzene',
                 'p-xylene', 'o-xylene', 'm-xylene', 'neopentane',
                 '1,3,5-TIPB'],
    'Diameter_A': [2.9, 3.6, 4.3, 6.0, 5.9, 5.8, 6.8, 6.8, 6.2, 8.5]
})

pore_windows = {
    'CHA (8-MR)': 3.8,
    'MFI (10-MR)': 5.5,
    'FAU (12-MR)': 7.4
}

In [ ]:
# Molecular size vs pore size comparison chart
fig, ax = plt.subplots(figsize=(12, 7))

y_positions = np.arange(len(molecules))
bar_height = 0.5

# Plot molecular diameter bars
bars = ax.barh(y_positions, molecules['Diameter_A'], height=bar_height,
               color=COLORS['skyblue'], edgecolor='black', linewidth=0.8,
               label='Critical molecular diameter')

# Add pore size vertical lines
pore_colors = [COLORS['vermillion'], COLORS['orange'], COLORS['blue']]
pore_styles = ['--', '-.', '-']
for (name, size), color, ls in zip(pore_windows.items(), pore_colors, pore_styles):
    ax.axvline(x=size, color=color, linestyle=ls, linewidth=2.5,
               label=f'{name}: {size} \u00c5')

# Color bars based on whether molecule fits in MFI
for i, (_, row) in enumerate(molecules.iterrows()):
    if row['Diameter_A'] <= pore_windows['MFI (10-MR)']:
        bars[i].set_facecolor(COLORS['green'])
        bars[i].set_alpha(0.7)
    elif row['Diameter_A'] <= pore_windows['FAU (12-MR)']:
        bars[i].set_facecolor(COLORS['orange'])
        bars[i].set_alpha(0.7)
    else:
        bars[i].set_facecolor(COLORS['vermillion'])
        bars[i].set_alpha(0.7)

ax.set_yticks(y_positions)
ax.set_yticklabels(molecules['Molecule'])
ax.set_xlabel('Critical Molecular Diameter (\u00c5)')
ax.set_title('Molecular Size vs. Zeolite Pore Window:\nReactant Shape Selectivity')
ax.set_xlim(0, 10)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3, axis='x')

# Annotations
ax.text(2.0, 9.5, 'Enters all\nzeolites', fontsize=9, color=COLORS['green'],
        ha='center', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.text(6.5, 9.5, 'Large pore\nonly', fontsize=9, color=COLORS['orange'],
        ha='center', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.text(9.0, 9.5, 'Excluded from\nall zeolites', fontsize=9,
        color=COLORS['vermillion'], ha='center',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

In [ ]:
def check_molecular_access(d_molecule, d_pore, tolerance=0.0):
    """
    Determine whether a molecule can enter a zeolite pore.

    Reactant shape selectivity: molecules with critical diameter
    larger than the pore window are excluded from the internal
    active sites.

    Parameters
    ----------
    d_molecule : float
        Critical molecular diameter (angstrom)
    d_pore : float
        Zeolite pore window diameter (angstrom)
    tolerance : float, optional
        Tolerance for borderline cases (angstrom). Default 0.

    Returns
    -------
    str
        'enters' if d_molecule < d_pore - tolerance,
        'restricted' if within tolerance,
        'excluded' if d_molecule > d_pore + tolerance
    """
    if d_molecule < d_pore - tolerance:
        return 'enters'
    elif d_molecule > d_pore + tolerance:
        return 'excluded'
    else:
        return 'restricted'


# Access matrix: which molecules can enter which zeolites?
print("Molecular Access Matrix (tolerance = 0.3 angstrom)")
print("=" * 70)
header = f"{'Molecule':<15}"
for name in pore_windows:
    header += f"{name:>18}"
print(header)
print("-" * 70)

for _, row in molecules.iterrows():
    line = f"{row['Molecule']:<15}"
    for name, pore_size in pore_windows.items():
        status = check_molecular_access(row['Diameter_A'], pore_size,
                                        tolerance=0.3)
        line += f"{status:>18}"
    print(line)

### The Three Types of Shape Selectivity

The access matrix above demonstrates **reactant selectivity**. The other two types are:

- **Product selectivity**: All products can form inside the pore, but compact isomers
  diffuse out faster. The classic example is para-xylene selectivity in MFI.
- **Transition-state selectivity**: Bulky transition states cannot form inside the pore,
  suppressing certain reaction pathways regardless of reactant or product size.

---
## Part 4: Configurational Diffusion in Micropores

Diffusion in zeolite micropores is fundamentally different from diffusion in mesopores or
macropores. When the pore diameter is comparable to the molecular diameter, molecules must
adopt specific orientations to squeeze through constrictions. This is called
**configurational diffusion**, and it follows an Arrhenius-like temperature dependence:

$$D_{\text{micro}} = D_0 \exp\left(-\frac{E_D}{RT}\right)$$

where $E_D$ is the activation energy for diffusion (10--50 kJ/mol), depending on how
tightly the molecule fits in the pore.

| Regime | Pore Size | $D_{\text{eff}}$ (m$^2$/s) | Rate-Limiting Factor |
|--------|-----------|------|----------------------|
| Molecular | >100 nm | $10^{-5}$ | Molecule-molecule collisions |
| Knudsen | 2--100 nm | $10^{-6}$ to $10^{-7}$ | Molecule-wall collisions |
| Configurational | <2 nm | $10^{-10}$ to $10^{-18}$ | Steric constraints |

In [ ]:
def bulk_diffusivity(T, P_atm, M_A, M_B, sigma_AB=3.5e-10, Omega=1.0):
    """
    Estimate gas-phase bulk (molecular) diffusivity using Chapman-Enskog.

    Simplified form for order-of-magnitude estimation.

    Parameters
    ----------
    T : float or array-like
        Temperature (K)
    P_atm : float
        Pressure (atm)
    M_A : float
        Molar mass of species A (g/mol)
    M_B : float
        Molar mass of species B (g/mol)
    sigma_AB : float, optional
        Collision diameter (m). Default 3.5e-10 m.
    Omega : float, optional
        Collision integral (dimensionless). Default 1.0.

    Returns
    -------
    float or ndarray
        Bulk diffusivity (m^2/s)
    """
    T = np.asarray(T, dtype=float)
    P_Pa = P_atm * 101325  # Convert atm to Pa
    M_reduced = 2 * M_A * M_B / ((M_A + M_B) * 1000)  # kg/mol
    # Simplified Chapman-Enskog
    D_AB = (1.858e-3 * T**1.5 * np.sqrt(1 / (M_A / 1000) + 1 / (M_B / 1000))
            / (P_Pa / 101325 * sigma_AB**2 * 1e20 * Omega)) * 1e-4
    # Rough estimate: scale as T^1.75 / P
    return 1e-5 * (T / 300)**1.75 / P_atm


def knudsen_diffusivity_m(d_pore_m, T, M_gmol):
    """
    Calculate Knudsen diffusivity.

    D_K = (d_pore / 3) * sqrt(8RT / (pi * M))

    Parameters
    ----------
    d_pore_m : float
        Pore diameter in meters
    T : float or array-like
        Temperature (K)
    M_gmol : float
        Molar mass (g/mol)

    Returns
    -------
    float or ndarray
        Knudsen diffusivity (m^2/s)
    """
    T = np.asarray(T, dtype=float)
    M_kg = M_gmol / 1000  # Convert to kg/mol
    return (d_pore_m / 3) * np.sqrt(8 * R * T / (np.pi * M_kg))


def configurational_diffusivity(T, D0, E_diff):
    """
    Calculate configurational (micropore) diffusivity.

    D_micro = D0 * exp(-E_diff / (R * T))

    Parameters
    ----------
    T : float or array-like
        Temperature (K)
    D0 : float
        Pre-exponential diffusivity factor (m^2/s)
    E_diff : float
        Activation energy for diffusion (J/mol)

    Returns
    -------
    float or ndarray
        Configurational diffusivity (m^2/s)

    Notes
    -----
    Typical values:
    - D0: 10^-8 to 10^-6 m^2/s
    - E_diff: 10-50 kJ/mol (depends on molecule-pore size ratio)
    """
    T = np.asarray(T, dtype=float)
    return D0 * np.exp(-E_diff / (R * T))

In [ ]:
# Compare three diffusion regimes across temperature
T_range = np.linspace(300, 800, 200)  # K
M_hexane = 86.18  # g/mol (n-hexane)

# Bulk diffusion: ~10^-5 m^2/s at 300K, scales with T^1.75
D_bulk = 1e-5 * (T_range / 300)**1.75

# Knudsen diffusion: mesopore d = 10 nm
D_knudsen = knudsen_diffusivity_m(10e-9, T_range, M_hexane)

# Configurational diffusion: n-hexane in MFI (E_D ~ 25 kJ/mol)
D_config = configurational_diffusivity(T_range, D0=1e-7, E_diff=25e3)

# Configurational: tighter fit, e.g. cyclohexane in MFI (E_D ~ 40 kJ/mol)
D_config_tight = configurational_diffusivity(T_range, D0=1e-7, E_diff=40e3)

fig, ax = plt.subplots(figsize=(10, 7))

ax.semilogy(T_range, D_bulk, color=COLORS['blue'], linestyle='-',
            linewidth=2.5, label=r'Bulk (molecular), $d_{\mathrm{pore}}$ > 100 nm')
ax.semilogy(T_range, D_knudsen, color=COLORS['orange'], linestyle='--',
            linewidth=2.5, label=r'Knudsen, $d_{\mathrm{pore}}$ = 10 nm')
ax.semilogy(T_range, D_config, color=COLORS['green'], linestyle='-.',
            linewidth=2.5,
            label='Configurational (MFI), $E_D$ = 25 kJ/mol')
ax.semilogy(T_range, D_config_tight, color=COLORS['vermillion'],
            linestyle=':', linewidth=2.5,
            label='Configurational (tight fit), $E_D$ = 40 kJ/mol')

# Add regime labels
ax.text(700, 5e-5, 'Bulk\nDiffusion', fontsize=11, color=COLORS['blue'],
        ha='center')
ax.text(700, 1e-7, 'Knudsen\nDiffusion', fontsize=11, color=COLORS['orange'],
        ha='center')

ax.set_xlabel('Temperature (K)')
ax.set_ylabel('Diffusivity, $D$ (m$^2$/s)')
ax.set_title('Diffusion Regimes: n-Hexane in Different Pore Environments')
ax.set_xlim(300, 800)
ax.set_ylim(1e-16, 1e-3)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

# Print representative values
print("\nDiffusivity Comparison at T = 500 K:")
print("=" * 50)
T_ref = 500
print(f"Bulk:            {1e-5 * (T_ref/300)**1.75:.2e} m^2/s")
print(f"Knudsen (10 nm): {knudsen_diffusivity_m(10e-9, T_ref, M_hexane):.2e} m^2/s")
print(f"Config (25 kJ):  {configurational_diffusivity(T_ref, 1e-7, 25e3):.2e} m^2/s")
print(f"Config (40 kJ):  {configurational_diffusivity(T_ref, 1e-7, 40e3):.2e} m^2/s")

In [ ]:
# Dramatic dependence of D on d_molecule/d_pore ratio
# This is the key insight: a small change in molecular size causes
# orders-of-magnitude change in diffusivity

d_ratio = np.linspace(0.1, 0.99, 200)  # d_molecule / d_pore

# Model: E_D increases steeply as molecule approaches pore size
# E_D = E_0 / (1 - lambda)^alpha, where lambda = d_mol/d_pore
# Simplified semi-empirical model for pedagogical illustration
E_0 = 5e3  # Base activation energy (J/mol) for small molecules
alpha = 2  # Exponent controlling steepness

E_D_array = E_0 / (1 - d_ratio)**alpha

# Diffusivity at 500 K
T_calc = 500  # K
D0_model = 1e-7  # m^2/s
D_array = D0_model * np.exp(-E_D_array / (R * T_calc))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: E_D vs ratio
ax1.plot(d_ratio, E_D_array / 1000, color=COLORS['blue'], linewidth=2.5)
ax1.set_xlabel(r'$d_{\mathrm{molecule}}$ / $d_{\mathrm{pore}}$')
ax1.set_ylabel(r'Diffusion Activation Energy, $E_D$ (kJ/mol)')
ax1.set_title('Steric Barrier Increases with Molecular Size')
ax1.set_xlim(0.1, 1.0)
ax1.set_ylim(0, 150)
ax1.grid(True, alpha=0.3)

# Mark representative molecules in MFI (d_pore = 5.5 A)
mfi_molecules = {
    'n-hexane': 4.3 / 5.5,
    'p-xylene': 5.8 / 5.5,
    'benzene': 5.9 / 5.5
}
for name, ratio_val in mfi_molecules.items():
    if ratio_val < 1.0:
        E_val = E_0 / (1 - ratio_val)**alpha / 1000
        ax1.plot(ratio_val, E_val, 'o', color=COLORS['vermillion'],
                 markersize=8, markeredgecolor='black')
        ax1.annotate(name, xy=(ratio_val, E_val),
                     xytext=(ratio_val - 0.12, E_val + 8),
                     fontsize=9)

# Right: D vs ratio (log scale)
ax2.semilogy(d_ratio, D_array, color=COLORS['vermillion'], linewidth=2.5)
ax2.set_xlabel(r'$d_{\mathrm{molecule}}$ / $d_{\mathrm{pore}}$')
ax2.set_ylabel('Diffusivity, $D$ (m$^2$/s) at 500 K')
ax2.set_title('Dramatic Drop in D as Molecule Approaches Pore Size')
ax2.set_xlim(0.1, 1.0)
ax2.set_ylim(1e-20, 1e-5)
ax2.grid(True, alpha=0.3, which='both')

# Reference lines for diffusion regimes
ax2.axhline(y=1e-10, color='gray', linestyle='--', alpha=0.5)
ax2.text(0.15, 2e-10, 'Configurational regime boundary', fontsize=9,
         color='gray')

plt.tight_layout()
plt.show()

### Key Observations

1. **Configurational diffusivity is 10$^{3}$--10$^{12}$ times slower** than Knudsen diffusion in mesopores
2. The dramatic drop in $D$ as $d_{\text{molecule}}/d_{\text{pore}} \to 1$ is the physical basis for shape selectivity
3. **Activated diffusion**: Unlike bulk or Knudsen diffusion (weak T dependence), configurational diffusion has strong Arrhenius-like T dependence
4. A small difference in molecular diameter (e.g., p-xylene vs o-xylene: 5.8 vs 6.8 angstrom) causes orders-of-magnitude difference in diffusivity

---
## Part 5: Zeolite Effectiveness Factor

Because micropore diffusivities are so small, zeolite catalysts can be severely
diffusion-limited even when the crystal is sub-micron sized. The effectiveness factor
framework from Chapter 4 applies, but the characteristic length is the **crystal radius**
(not pellet radius).

For spherical geometry:
$$\eta = \frac{3}{\phi^2}(\phi \coth \phi - 1)$$

where $\phi = R_{\text{crystal}} \sqrt{k / D_{\text{micro}}}$

In [ ]:
def effectiveness_sphere(phi):
    """
    Calculate effectiveness factor for spherical geometry.

    eta = 3 * (phi * coth(phi) - 1) / phi^2

    Parameters
    ----------
    phi : float or array-like
        Thiele modulus (dimensionless)

    Returns
    -------
    float or ndarray
        Effectiveness factor eta, range (0, 1]

    Notes
    -----
    Limiting cases:
    - phi << 1: eta ~ 1 (kinetic control)
    - phi >> 1: eta ~ 3/phi (diffusion control)
    """
    phi = np.asarray(phi, dtype=float)
    with np.errstate(divide='ignore', invalid='ignore'):
        coth_phi = 1.0 / np.tanh(phi)
        eta = 3 * (phi * coth_phi - 1) / phi**2
        eta = np.where(phi < 1e-10, 1.0, eta)
    return eta


def effectiveness_slab(phi):
    """
    Calculate effectiveness factor for slab (flat plate) geometry.

    eta = tanh(phi) / phi

    Parameters
    ----------
    phi : float or array-like
        Thiele modulus (dimensionless)

    Returns
    -------
    float or ndarray
        Effectiveness factor eta, range (0, 1]
    """
    phi = np.asarray(phi, dtype=float)
    with np.errstate(divide='ignore', invalid='ignore'):
        eta = np.where(phi < 1e-10, 1.0, np.tanh(phi) / phi)
    return eta

In [ ]:
# Effectiveness factor vs crystal radius for different D_micro values
# Worked example from lecture notes: k = 10 s^-1
k_intrinsic = 10  # s^-1 (first-order rate constant at 500 C)

R_crystal_um = np.logspace(-2, 1, 300)  # Crystal radius in micrometers
R_crystal_m = R_crystal_um * 1e-6  # Convert to meters

# Three diffusivity scenarios
D_values = {
    r'$D_{\mathrm{micro}}$ = $10^{-10}$ m$^2$/s (fast)': 1e-10,
    r'$D_{\mathrm{micro}}$ = $10^{-11}$ m$^2$/s (medium)': 1e-11,
    r'$D_{\mathrm{micro}}$ = $10^{-12}$ m$^2$/s (slow)': 1e-12
}

colors_list = [COLORS['blue'], COLORS['orange'], COLORS['vermillion']]
linestyles_list = ['--', '-.', '-']

fig, ax = plt.subplots(figsize=(10, 7))

for (label, D_micro), color, ls in zip(D_values.items(), colors_list, linestyles_list):
    phi = R_crystal_m * np.sqrt(k_intrinsic / D_micro)
    eta = effectiveness_sphere(phi)
    ax.semilogx(R_crystal_um, eta, color=color, linestyle=ls,
                linewidth=2.5, label=label)

# Highlight eta = 0.9 threshold
ax.axhline(y=0.9, color='gray', linestyle=':', linewidth=1.5, alpha=0.7)
ax.text(0.012, 0.92, r'$\eta$ = 0.9 threshold', fontsize=10, color='gray')

# Typical crystal size ranges
ax.axvspan(0.05, 0.1, alpha=0.1, color=COLORS['green'],
           label='Nano-zeolite range')
ax.axvspan(0.5, 5, alpha=0.1, color=COLORS['purple'],
           label='Conventional crystal range')

ax.set_xlabel(r'Crystal Radius ($\mu$m)')
ax.set_ylabel(r'Effectiveness Factor, $\eta$')
ax.set_title(f'Zeolite Effectiveness Factor vs. Crystal Size\n'
             f'$k$ = {k_intrinsic} s$^{{-1}}$ (first-order, 500 $^\\circ$C)')
ax.set_xlim(0.01, 10)
ax.set_ylim(0, 1.05)
ax.legend(loc='lower left', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Worked example from lecture notes: H-Y zeolite cracking
print("Worked Example: Thiele Modulus for H-Y Zeolite Crystal")
print("=" * 55)

R_crystal = 0.5e-6  # 0.5 micrometer = 500 nm
k_cracking = 10  # s^-1
D_micro = 5e-12  # m^2/s

phi = R_crystal * np.sqrt(k_cracking / D_micro)
eta = effectiveness_sphere(np.array([phi]))[0]

print(f"Crystal radius: R = {R_crystal*1e6:.1f} um = {R_crystal*1e9:.0f} nm")
print(f"Intrinsic rate constant: k = {k_cracking} s^-1")
print(f"Micropore diffusivity: D_micro = {D_micro:.0e} m^2/s")
print(f"")
print(f"Thiele modulus: phi = {phi:.2f}")
print(f"Effectiveness factor: eta = {eta:.3f}")
print()

# Find maximum crystal radius for eta > 0.9
# For sphere: eta = 0.9 at phi ~ 0.94
phi_target = 0.94
R_max = phi_target / np.sqrt(k_cracking / D_micro)
print(f"For eta > 0.9: need phi < {phi_target:.2f}")
print(f"Maximum crystal radius: R < {R_max*1e6:.2f} um = {R_max*1e9:.0f} nm")

In [ ]:
# Comparison: zeolite micropore vs mesoporous support
# Same intrinsic kinetics, different diffusivities
phi_range = np.logspace(-2, 2, 500)

eta_sphere = effectiveness_sphere(phi_range)
eta_slab = effectiveness_slab(phi_range)

fig, ax = plt.subplots(figsize=(10, 7))

ax.loglog(phi_range, eta_sphere, color=COLORS['blue'], linestyle='-',
          linewidth=2.5, label='Sphere geometry')
ax.loglog(phi_range, eta_slab, color=COLORS['orange'], linestyle='--',
          linewidth=2.5, label='Slab geometry')

# Asymptotic limits
phi_high = phi_range[phi_range > 3]
ax.loglog(phi_high, 3 / phi_high, color=COLORS['blue'], linestyle=':',
          linewidth=1.5, alpha=0.5, label=r'Sphere asymptote: $3/\phi$')
ax.loglog(phi_high, 1 / phi_high, color=COLORS['orange'], linestyle=':',
          linewidth=1.5, alpha=0.5, label=r'Slab asymptote: $1/\phi$')

# Mark representative operating points
# Zeolite: D = 10^-12, k = 10, R = 0.5 um -> phi = 0.71
phi_zeolite = 0.71
eta_zeolite = effectiveness_sphere(np.array([phi_zeolite]))[0]
ax.plot(phi_zeolite, eta_zeolite, 's', color=COLORS['vermillion'],
        markersize=14, markeredgecolor='black', markeredgewidth=2,
        label=rf'Zeolite crystal ($\phi$ = {phi_zeolite:.2f})', zorder=5)

# Mesoporous support: D = 10^-7, k = 10, R = 1 mm -> phi = 10
phi_meso = 10
eta_meso = effectiveness_sphere(np.array([phi_meso]))[0]
ax.plot(phi_meso, eta_meso, 'D', color=COLORS['green'],
        markersize=14, markeredgecolor='black', markeredgewidth=2,
        label=rf'Meso pellet ($\phi$ = {phi_meso:.0f})', zorder=5)

# Regime labels
ax.text(0.03, 0.85, 'Kinetic\nControl', fontsize=12, ha='center',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax.text(30, 0.05, 'Diffusion\nControl', fontsize=12, ha='center',
        bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.3))

ax.set_xlabel(r'Thiele Modulus, $\phi$')
ax.set_ylabel(r'Effectiveness Factor, $\eta$')
ax.set_title('Effectiveness Factor: Sphere vs. Slab Geometry')
ax.set_xlim(0.01, 100)
ax.set_ylim(0.01, 1.5)
ax.legend(loc='lower left', fontsize=10)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

### Key Observations

1. **Small crystals are essential**: Even at R = 0.5 um, the zeolite is near the kinetic-diffusion transition
2. **Crystal size is the primary design lever**: Because D_micro is fixed by the framework and molecule, only R can be adjusted
3. **Sphere vs slab**: Sphere geometry gives slightly higher eta at the same phi (3/phi vs 1/phi asymptote) because of the higher surface-to-volume ratio
4. **Nano-zeolites** (<100 nm) can maintain kinetic control even for fast reactions with slow diffusion

---
## Part 6: Para-Xylene Selectivity Case Study

The selective production of para-xylene in MFI (ZSM-5) zeolite is the textbook example
of **product shape selectivity**. All three xylene isomers can form inside the zeolite
at near-equilibrium ratios, but para-xylene diffuses out much faster than ortho- or
meta-xylene.

**Diffusivities at 400 degrees C** (from lecture notes):
- para-xylene: $D_p = 1 \times 10^{-11}$ m$^2$/s (kinetic diameter 5.8 angstrom)
- ortho-xylene: $D_o = 5 \times 10^{-14}$ m$^2$/s (kinetic diameter 6.8 angstrom)
- meta-xylene: $D_m = 2 \times 10^{-13}$ m$^2$/s (kinetic diameter 6.8 angstrom)

In [ ]:
# Xylene isomer diffusivities in MFI at 400 C
xylene_data = {
    'p-xylene': {'D': 1e-11, 'd_kinetic': 5.8, 'color': COLORS['blue']},
    'o-xylene': {'D': 5e-14, 'd_kinetic': 6.8, 'color': COLORS['vermillion']},
    'm-xylene': {'D': 2e-13, 'd_kinetic': 6.8, 'color': COLORS['orange']}
}

# Diffusivity ratios
D_p = xylene_data['p-xylene']['D']
D_o = xylene_data['o-xylene']['D']
D_m = xylene_data['m-xylene']['D']

print("Xylene Diffusivities in MFI (ZSM-5) at 400 C")
print("=" * 55)
print(f"p-xylene: D = {D_p:.0e} m^2/s  (d_kin = 5.8 A)")
print(f"o-xylene: D = {D_o:.0e} m^2/s  (d_kin = 6.8 A)")
print(f"m-xylene: D = {D_m:.0e} m^2/s  (d_kin = 6.8 A)")
print()
print(f"D_p / D_o = {D_p/D_o:.0f}")
print(f"D_p / D_m = {D_p/D_m:.0f}")
print()

# Characteristic diffusion times: t ~ L^2 / D
L_crystal = 0.2e-6  # 0.2 um crystal half-thickness
print(f"Characteristic diffusion times (L = {L_crystal*1e6:.1f} um):")
print(f"p-xylene: t = {L_crystal**2 / D_p * 1000:.1f} ms")
print(f"o-xylene: t = {L_crystal**2 / D_o * 1000:.0f} ms")
print(f"m-xylene: t = {L_crystal**2 / D_m * 1000:.0f} ms")

In [ ]:
# Simple selectivity model:
# All isomers form at equilibrium rates inside the crystal.
# Selectivity is determined by relative diffusion rates out.
# In steady state, product flux ~ D_i * (C_interior - C_exterior)
# With fast re-equilibration inside, selectivity ~ D_i / sum(D_j)

def product_selectivity(D_dict, equilibrium_fractions=None):
    """
    Estimate product selectivity based on diffusion-limited removal.

    Assumes all products form at equilibrium inside the zeolite and
    that the product mix exiting the crystal is proportional to each
    isomer's diffusivity (fast re-equilibration limit).

    Parameters
    ----------
    D_dict : dict
        Dictionary mapping product names to their diffusivities
    equilibrium_fractions : dict, optional
        Equilibrium mole fractions. If None, assumes equal fractions.

    Returns
    -------
    dict
        Dictionary mapping product names to selectivity fractions
    """
    if equilibrium_fractions is None:
        equilibrium_fractions = {k: 1.0 / len(D_dict) for k in D_dict}

    # Product flux proportional to D_i * x_eq_i
    fluxes = {k: D_dict[k] * equilibrium_fractions[k] for k in D_dict}
    total_flux = sum(fluxes.values())

    selectivities = {k: v / total_flux for k, v in fluxes.items()}
    return selectivities


# Thermodynamic equilibrium at 400 C: ~24% para, ~52% meta, ~24% ortho
equil = {'p-xylene': 0.24, 'o-xylene': 0.24, 'm-xylene': 0.52}
D_dict = {'p-xylene': D_p, 'o-xylene': D_o, 'm-xylene': D_m}

selectivity = product_selectivity(D_dict, equil)

print("\nPara-Xylene Selectivity Analysis")
print("=" * 55)
print(f"{'Isomer':<15} {'Equilibrium':>12} {'Shape-selective':>18}")
print("-" * 50)
for name in D_dict:
    print(f"{name:<15} {equil[name]:>12.0%} {selectivity[name]:>18.1%}")
print()
print(f"Enhancement of p-xylene: {selectivity['p-xylene']/equil['p-xylene']:.1f}x")

In [ ]:
# Visualize the selectivity comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

isomers = list(equil.keys())
equil_vals = [equil[k] * 100 for k in isomers]
select_vals = [selectivity[k] * 100 for k in isomers]
bar_colors = [xylene_data[k]['color'] for k in isomers]

x = np.arange(len(isomers))
width = 0.35

# Left: Bar comparison
bars1 = ax1.bar(x - width/2, equil_vals, width, color='lightgray',
               edgecolor='black', linewidth=1.2, label='Thermodynamic equilibrium')
bars2 = ax1.bar(x + width/2, select_vals, width, color=bar_colors,
               edgecolor='black', linewidth=1.2, label='Shape-selective (MFI)')

ax1.set_xticks(x)
ax1.set_xticklabels(['para', 'ortho', 'meta'])
ax1.set_ylabel('Product Selectivity (%)')
ax1.set_title('Xylene Isomer Selectivity')
ax1.legend(loc='upper right')
ax1.set_ylim(0, 100)
ax1.grid(True, alpha=0.3, axis='y')

# Add values on bars
for bar, val in zip(bars1, equil_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.0f}%', ha='center', va='bottom', fontsize=10)
for bar, val in zip(bars2, select_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.0f}%', ha='center', va='bottom', fontsize=10,
             fontweight='bold')

# Right: Diffusivity bar chart (log scale)
D_vals = [xylene_data[k]['D'] for k in isomers]
bars3 = ax2.bar(isomers, D_vals, color=bar_colors,
               edgecolor='black', linewidth=1.2)
ax2.set_yscale('log')
ax2.set_ylabel('Diffusivity (m$^2$/s)')
ax2.set_title(r'Diffusivities in MFI at 400 $^\circ$C')
ax2.grid(True, alpha=0.3, which='both', axis='y')

# Add ratio annotations
ax2.annotate(f'{D_p/D_o:.0f}x', xy=(0.5, np.sqrt(D_p*D_o)),
             fontsize=14, ha='center', fontweight='bold', color=COLORS['green'])

plt.tight_layout()
plt.show()

In [ ]:
# Effect of crystal size on para-selectivity
# Larger crystals = longer diffusion path = stronger shape selectivity
# but also lower overall activity (lower eta)

# Model: para-selectivity increases with crystal size because
# bulky isomers spend more time inside and re-equilibrate more
# Simplified model: selectivity approaches diffusion-limited value
# as crystal size increases

crystal_sizes_nm = np.logspace(1, 4, 100)  # 10 nm to 10 um
crystal_sizes_m = crystal_sizes_nm * 1e-9

# For each crystal size, compute phi and eta for each isomer
k_isom = 1.0  # s^-1 isomerization rate constant (assumed)

phi_p = crystal_sizes_m * np.sqrt(k_isom / D_p)
phi_o = crystal_sizes_m * np.sqrt(k_isom / D_o)
phi_m = crystal_sizes_m * np.sqrt(k_isom / D_m)

eta_p = effectiveness_sphere(phi_p)
eta_o = effectiveness_sphere(phi_o)
eta_m = effectiveness_sphere(phi_m)

# Para-selectivity = eta_p / (eta_p + eta_o + eta_m) (simplified model)
para_selectivity = eta_p / (eta_p + eta_o + eta_m) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: eta for each isomer
ax1.semilogx(crystal_sizes_nm, eta_p, color=COLORS['blue'],
             linewidth=2.5, label='p-xylene')
ax1.semilogx(crystal_sizes_nm, eta_o, color=COLORS['vermillion'],
             linewidth=2.5, linestyle='--', label='o-xylene')
ax1.semilogx(crystal_sizes_nm, eta_m, color=COLORS['orange'],
             linewidth=2.5, linestyle='-.', label='m-xylene')

ax1.set_xlabel('Crystal Radius (nm)')
ax1.set_ylabel(r'Effectiveness Factor, $\eta$')
ax1.set_title(r'$\eta$ for Each Xylene Isomer in MFI')
ax1.legend(loc='lower left')
ax1.set_ylim(0, 1.05)
ax1.grid(True, alpha=0.3)

# Right: para-selectivity vs crystal size
ax2.semilogx(crystal_sizes_nm, para_selectivity, color=COLORS['blue'],
             linewidth=2.5)
ax2.axhline(y=24, color='gray', linestyle=':', linewidth=1.5)
ax2.text(15, 25, 'Thermodynamic limit (24%)', fontsize=10, color='gray')
ax2.axhline(y=90, color=COLORS['green'], linestyle='--', linewidth=1.5)
ax2.text(15, 91, 'Commercial target (>90%)', fontsize=10, color=COLORS['green'])

ax2.set_xlabel('Crystal Radius (nm)')
ax2.set_ylabel('Para-Xylene Selectivity (%)')
ax2.set_title('Shape Selectivity vs. Crystal Size')
ax2.set_ylim(0, 100)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Key Observations

1. **Small crystals** (<100 nm): All isomers escape easily, selectivity approaches thermodynamic equilibrium (~24% para)
2. **Intermediate crystals** (100--1000 nm): ortho/meta start to be diffusion-limited while para is still kinetically controlled
3. **Large crystals** (>1 um): Strong shape selectivity, but overall activity drops because even para-xylene becomes diffusion-limited
4. **Optimal design**: Select crystal size that maximizes para-selectivity while maintaining acceptable activity

The 200-fold diffusivity ratio between para- and ortho-xylene is the molecular basis for ZSM-5 achieving >90% para-selectivity in industrial xylene isomerization.

---
## Part 7: Interactive Parameter Exploration

Modify the parameters below to explore how zeolite design choices affect catalyst performance.

In [ ]:
# =========================================================================
# ADJUSTABLE PARAMETERS - Modify these to explore different scenarios
# =========================================================================

# Zeolite properties
crystal_radius_um = 0.5     # Crystal radius in micrometers
D_micro_m2s = 5e-12         # Micropore diffusivity (m^2/s)
k_intrinsic_s = 10          # Intrinsic rate constant (s^-1)
si_al_ratio_val = 25        # Si/Al molar ratio
T_atoms_uc = 96             # T-atoms per unit cell (MFI = 96)
n_O_uc = 192                # Oxygens per unit cell (MFI = 192)

# =========================================================================
# CALCULATIONS (do not modify)
# =========================================================================

R_m = crystal_radius_um * 1e-6
phi_val = R_m * np.sqrt(k_intrinsic_s / D_micro_m2s)
eta_val = effectiveness_sphere(np.array([phi_val]))[0]
acid_result = acid_site_density(si_al_ratio_val, T_atoms_uc, n_O_uc)

# Weisz-Prater criterion
WP = phi_val**2 * eta_val

print("Zeolite Catalyst Performance Summary")
print("=" * 55)
print(f"Crystal radius:         {crystal_radius_um:.2f} um")
print(f"Micropore diffusivity:  {D_micro_m2s:.1e} m^2/s")
print(f"Intrinsic rate const:   {k_intrinsic_s:.1f} s^-1")
print(f"Si/Al ratio:            {si_al_ratio_val}")
print(f"")
print(f"Thiele modulus:         phi = {phi_val:.3f}")
print(f"Effectiveness factor:   eta = {eta_val:.4f}")
print(f"Observed rate / max:    {eta_val:.1%}")
print(f"Weisz-Prater:           WP = {WP:.3f}", end='')
if WP < 0.3:
    print(" (negligible diffusion limitation)")
elif WP < 3:
    print(" (transition regime)")
else:
    print(" (severe diffusion limitation)")
print(f"")
print(f"Acid site density:      {acid_result['acid_density_mmol_g']:.2f} mmol/g")
print(f"Al per unit cell:       {acid_result['n_Al']:.1f}")

### Key Takeaways — Part I: Zeolites

- Framework type (FAU, MFI, CHA) determines pore size and shape selectivity
- Three types of shape selectivity: reactant, product, and transition-state
- Configurational diffusion in micropores is orders of magnitude slower than bulk diffusion
- Acid site density depends on Si/Al ratio: fewer Al = fewer but stronger acid sites

> **Session Break (suggested):** Good stopping point between zeolites and carbon nanotubes. Save your work before continuing.

---
## Part 8: Carbon Nanotubes

From the confined micropores of zeolites (3--8 Å), we now turn to carbon nanotubes where smooth graphitic walls enable *enhanced* transport -- diffusivities 10--10,000× faster than Knudsen predictions.

In [ ]:
# NOTE: This local cnt_diameter() returns a dict with diameter_nm,
# chirality, electronic, and chiral_angle_deg keys.
# utils.py has a different signature: cnt_diameter(n, m) returns
# (diameter, chirality) as a tuple, without electronic character.

def cnt_diameter(n, m, a=0.246):
    """
    Calculate CNT diameter and classify chirality and electronic character.

    The diameter is computed from the chiral indices using:
    d = (a / pi) * sqrt(n^2 + n*m + m^2)
    where a = 0.246 nm is the graphene lattice constant.

    Parameters
    ----------
    n : int
        First chiral index (n >= m >= 0)
    m : int
        Second chiral index
    a : float, optional
        Graphene lattice constant (nm). Default 0.246 nm.

    Returns
    -------
    dict
        Dictionary with keys:
        - 'diameter_nm': Nanotube diameter (nm)
        - 'chirality': 'armchair', 'zigzag', or 'chiral'
        - 'electronic': 'metallic' or 'semiconducting'
        - 'chiral_angle_deg': Chiral angle in degrees

    Notes
    -----
    The lattice constant a = 0.246 nm derives from the C-C bond length
    (0.142 nm) in the sp2 network: a = 0.142 * sqrt(3).
    Electronic character: metallic if (n - m) % 3 == 0, else semiconducting.
    """
    d = (a / np.pi) * np.sqrt(n**2 + n * m + m**2)

    # Chirality classification
    if n == m:
        chirality = 'armchair'
    elif m == 0:
        chirality = 'zigzag'
    else:
        chirality = 'chiral'

    # Electronic character
    if (n - m) % 3 == 0:
        electronic = 'metallic'
    else:
        electronic = 'semiconducting'

    # Chiral angle
    theta = np.degrees(np.arctan(np.sqrt(3) * m / (2 * n + m)))

    return {
        'diameter_nm': round(d, 3),
        'chirality': chirality,
        'electronic': electronic,
        'chiral_angle_deg': round(theta, 1)
    }

In [ ]:
# Verify with worked example from lecture notes: (10, 5)
test_cases = [(10, 10), (10, 0), (10, 5), (5, 5), (6, 0), (9, 0)]

print("CNT Diameter Calculations")
print("=" * 70)
print(f"{'(n,m)':<10} {'d (nm)':>8} {'Chirality':<12} {'Electronic':<16} {'Angle':>6}")
print("-" * 70)

for n, m in test_cases:
    result = cnt_diameter(n, m)
    print(f"({n},{m}){'':<5} {result['diameter_nm']:>8.3f} "
          f"{result['chirality']:<12} {result['electronic']:<16} "
          f"{result['chiral_angle_deg']:>5.1f}")

print()
print("Verification:")
print(f"  (10,10) armchair: d = {cnt_diameter(10,10)['diameter_nm']:.3f} nm "
      f"(expected: ~1.356 nm)")
print(f"  (10,0)  zigzag:   d = {cnt_diameter(10,0)['diameter_nm']:.3f} nm "
      f"(expected: ~0.783 nm)")

In [ ]:
# Create the (n,m) map showing chirality and electronic character
fig, ax = plt.subplots(figsize=(12, 10))

n_max = 15

for n in range(0, n_max + 1):
    for m in range(0, n + 1):  # Convention: n >= m
        if n == 0 and m == 0:
            continue
        result = cnt_diameter(n, m)
        d = result['diameter_nm']

        # Color by electronic character
        if result['electronic'] == 'metallic':
            color = COLORS['blue']
            marker = 'o'
        else:
            color = COLORS['orange']
            marker = 's'

        # Size proportional to diameter
        size = 30 + d * 60

        # Highlight special types with edge color
        if result['chirality'] == 'armchair':
            edgecolor = COLORS['vermillion']
            linewidth = 2.5
        elif result['chirality'] == 'zigzag':
            edgecolor = COLORS['green']
            linewidth = 2.5
        else:
            edgecolor = 'gray'
            linewidth = 0.5

        ax.scatter(n, m, s=size, c=color, marker=marker,
                   edgecolors=edgecolor, linewidths=linewidth,
                   alpha=0.8, zorder=3)

# Add diameter labels for selected tubes
labeled = [(10, 10), (10, 0), (5, 5), (10, 5), (6, 0), (7, 0)]
for n, m in labeled:
    d = cnt_diameter(n, m)['diameter_nm']
    ax.annotate(f'{d:.2f} nm', xy=(n, m), xytext=(n + 0.4, m + 0.4),
                fontsize=8, color='black',
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

# Armchair line (n = m)
ax.plot([0, n_max], [0, n_max], '--', color=COLORS['vermillion'],
        linewidth=1.5, alpha=0.5)
ax.text(n_max - 1, n_max - 0.5, 'Armchair\n$(n,n)$',
        fontsize=10, color=COLORS['vermillion'])

# Zigzag line (m = 0)
ax.axhline(y=0, color=COLORS['green'], linestyle='--',
           linewidth=1.5, alpha=0.5)
ax.text(n_max - 1, 0.5, 'Zigzag $(n,0)$',
        fontsize=10, color=COLORS['green'])

# Legend
legend_elements = [
    plt.scatter([], [], s=80, c=COLORS['blue'], marker='o',
                edgecolors='gray', label='Metallic'),
    plt.scatter([], [], s=80, c=COLORS['orange'], marker='s',
                edgecolors='gray', label='Semiconducting'),
    plt.scatter([], [], s=80, c='white', marker='o',
                edgecolors=COLORS['vermillion'], linewidths=2.5,
                label='Armchair edge'),
    plt.scatter([], [], s=80, c='white', marker='o',
                edgecolors=COLORS['green'], linewidths=2.5,
                label='Zigzag edge')
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=11)

ax.set_xlabel('Chiral Index $n$')
ax.set_ylabel('Chiral Index $m$')
ax.set_title('CNT $(n, m)$ Map: Chirality and Electronic Character\n'
             r'(point size $\propto$ diameter)')
ax.set_xlim(-0.5, n_max + 0.5)
ax.set_ylim(-0.5, n_max + 0.5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

# Statistics
total = 0
metallic_count = 0
for n in range(1, n_max + 1):
    for m in range(0, n + 1):
        total += 1
        if (n - m) % 3 == 0:
            metallic_count += 1

print(f"\nOf {total} distinct (n,m) pairs with n <= {n_max}:")
print(f"  Metallic: {metallic_count} ({metallic_count/total:.0%})")
print(f"  Semiconducting: {total - metallic_count} "
      f"({(total-metallic_count)/total:.0%})")
print("  (Approaches 1/3 metallic for large n)")

### Key Observations

1. **Diameter increases** with both $n$ and $m$: larger indices give larger tubes
2. **One-third of tubes are metallic** -- those where $(n-m) \bmod 3 = 0$
3. **Armchair tubes** ($n = n$) are always metallic
4. **Zigzag tubes** ($n, 0$) are metallic only if $n$ is divisible by 3
5. Typical SWCNT diameters: 0.4--2 nm; MWCNT inner diameters: 5--100 nm

---
## Part 9: SWCNT vs MWCNT Comparison

| Property | SWCNT | MWCNT |
|----------|-------|-------|
| Diameter | 0.4--2 nm | 5--100 nm |
| Number of walls | 1 | 2--50+ |
| Surface area | 300--900 m$^2$/g | 200--400 m$^2$/g |
| Electronic properties | Tunable (metallic/semi) | Mixed |
| Synthesis control | More difficult | Easier |
| Cost | Higher | Lower |

In [ ]:
# SWCNT vs MWCNT property comparison
properties = {
    'Typical diameter (nm)': {'SWCNT': (0.7, 2.0), 'MWCNT': (5, 100)},
    'Surface area (m$^2$/g)': {'SWCNT': (300, 900), 'MWCNT': (200, 400)},
    'Inner diameter for transport (nm)': {'SWCNT': (0.7, 2.0), 'MWCNT': (3, 50)},
}

# Surface area calculation for ideal SWCNT
# Geometric SA of a cylinder: 2*pi*r*L per unit length
# For single graphene sheet: SA = 2630 m^2/g (theoretical)
# SWCNT accessible: inner + outer surface ~ 2 * graphene = ~1315 m^2/g max
# In practice, bundling reduces accessible SA

print("Surface Area Estimation for SWCNTs")
print("=" * 50)
print(f"Theoretical graphene: 2630 m^2/g (both sides)")
print(f"Ideal open SWCNT: ~1315 m^2/g (inner + outer)")
print(f"Practical (bundled): 300-500 m^2/g")
print(f"Debundled/open-ended: 600-900 m^2/g")
print()

# Diameter range visualization
fig, ax = plt.subplots(figsize=(10, 5))

# SWCNT diameter range from (n,m) calculations
swcnt_diameters = []
for n in range(3, 20):
    for m in range(0, n + 1):
        d = cnt_diameter(n, m)['diameter_nm']
        if 0.4 <= d <= 2.5:
            swcnt_diameters.append(d)

ax.hist(swcnt_diameters, bins=30, color=COLORS['blue'], alpha=0.6,
        edgecolor='black', label='SWCNT (from chiral indices)')

# MWCNT inner diameter range (simplified distribution)
np.random.seed(42)
mwcnt_diameters = np.random.lognormal(mean=np.log(15), sigma=0.6, size=500)
mwcnt_diameters = mwcnt_diameters[(mwcnt_diameters > 3) & (mwcnt_diameters < 80)]
ax.hist(mwcnt_diameters, bins=30, color=COLORS['orange'], alpha=0.6,
        edgecolor='black', label='MWCNT inner diameter (typical)')

ax.set_xlabel('Diameter (nm)')
ax.set_ylabel('Count')
ax.set_title('Diameter Distribution: SWCNT vs MWCNT')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Part 10: Knudsen vs Enhanced Diffusivity

In CNT channels, the Knudsen number $Kn = \lambda / d_{\text{pore}} \gg 1$ for most
conditions, placing transport in the Knudsen regime. However, the atomically smooth
graphitic walls enable **specular reflection** rather than the diffuse reflection
assumed in classical Knudsen theory.

The enhancement factor $F$ connects the specularity parameter $\sigma$ to the diffusivity:
$$F = \frac{1 + \sigma}{1 - \sigma}$$

| $\sigma$ | $F$ | Surface type |
|----------|-----|--------------|
| 0 | 1 | Completely rough (classical Knudsen) |
| 0.5 | 3 | Moderately smooth |
| 0.9 | 19 | Pristine CNT |
| 0.95 | 39 | High-quality SWCNT |
| 0.99 | 199 | Near-perfect specular |

In [ ]:
# NOTE: This local knudsen_diffusivity(d_pore_nm, T, M_gmol) uses
# (d_pore_nm, T, M_gmol) argument order. utils.py has a different
# signature: knudsen_diffusivity(T, M_gmol, d_pore_nm).

def knudsen_diffusivity(d_pore_nm, T, M_gmol):
    """
    Calculate classical Knudsen diffusivity.

    D_K = (d_pore / 3) * sqrt(8*R*T / (pi*M))

    Parameters
    ----------
    d_pore_nm : float or array-like
        Pore (channel) diameter in nanometers
    T : float or array-like
        Temperature (K)
    M_gmol : float
        Molar mass (g/mol)

    Returns
    -------
    float or ndarray
        Knudsen diffusivity (m^2/s)

    Notes
    -----
    Uses SI units internally: d in m, M in kg/mol.
    Typical output range: 10^-7 to 10^-5 m^2/s for gases in nanopores.
    """
    d_m = np.asarray(d_pore_nm, dtype=float) * 1e-9  # nm to m
    T = np.asarray(T, dtype=float)
    M_kg = M_gmol / 1000  # g/mol to kg/mol
    return (d_m / 3) * np.sqrt(8 * R * T / (np.pi * M_kg))


def enhanced_diffusivity(d_pore_nm, T, M_gmol, F):
    """
    Calculate enhanced CNT diffusivity.

    D_CNT = F * D_K

    Parameters
    ----------
    d_pore_nm : float
        CNT inner diameter (nm)
    T : float or array-like
        Temperature (K)
    M_gmol : float
        Molar mass (g/mol)
    F : float
        Enhancement factor (1 = classical, 10-10000 for CNTs)

    Returns
    -------
    float or ndarray
        Enhanced diffusivity (m^2/s)

    Notes
    -----
    F = (1 + sigma) / (1 - sigma), where sigma is specularity parameter.
    """
    D_K = knudsen_diffusivity(d_pore_nm, T, M_gmol)
    return F * D_K


def specularity_to_F(sigma):
    """
    Convert specularity parameter to enhancement factor.

    Parameters
    ----------
    sigma : float
        Specularity parameter (0 = diffuse, 1 = specular)

    Returns
    -------
    float
        Enhancement factor F
    """
    return (1 + sigma) / (1 - sigma)

In [ ]:
# Worked example from lecture notes: H2 in 2 nm CNT at 400 K
d_pore = 2.0  # nm
T_calc = 400  # K
M_H2 = 2.0  # g/mol
F_example = 100  # Enhancement factor

D_K = knudsen_diffusivity(d_pore, T_calc, M_H2)
D_CNT = enhanced_diffusivity(d_pore, T_calc, M_H2, F_example)

print("Worked Example: H2 Diffusion in 2 nm CNT at 400 K")
print("=" * 55)
print(f"Pore diameter: {d_pore} nm")
print(f"Temperature: {T_calc} K")
print(f"Molar mass (H2): {M_H2} g/mol")
print(f"Enhancement factor: F = {F_example}")
print()
print(f"Classical Knudsen:  D_K   = {D_K:.2e} m^2/s")
print(f"Enhanced CNT:       D_CNT = {D_CNT:.2e} m^2/s")
print(f"Ratio D_CNT / D_K = {D_CNT/D_K:.0f}")

**Concept Check:** The Knudsen diffusivity for H$_2$ in a 2 nm pore is about $10^{-5}$ m$^2$/s. Before seeing the enhancement, predict: would a real measurement in a CNT of the same diameter give $D$ closer to $10^{-5}$, $10^{-3}$, or $10^{-1}$ m$^2$/s?

In [ ]:
# Diffusivity vs temperature for different enhancement factors
T_range = np.linspace(300, 800, 200)
d_cnt = 5.0  # nm (MWCNT inner diameter)
M_gas = 28.0  # g/mol (N2 as a representative gas)

F_values = [1, 10, 100, 1000]
colors_F = [COLORS['blue'], COLORS['orange'], COLORS['green'],
            COLORS['vermillion']]
linestyles_F = ['-', '--', '-.', ':']

fig, ax = plt.subplots(figsize=(10, 7))

for F, color, ls in zip(F_values, colors_F, linestyles_F):
    D_vals = enhanced_diffusivity(d_cnt, T_range, M_gas, F)
    label = f'F = {F}' if F > 1 else 'Classical Knudsen (F = 1)'
    ax.semilogy(T_range, D_vals, color=color, linestyle=ls,
                linewidth=2.5, label=label)

# Reference: bulk gas diffusivity
D_bulk_approx = 1e-5 * (T_range / 300)**1.75
ax.semilogy(T_range, D_bulk_approx, color='gray', linestyle='-',
            linewidth=1.5, alpha=0.5, label='Bulk gas diffusion (approx)')

ax.set_xlabel('Temperature (K)')
ax.set_ylabel('Diffusivity, $D$ (m$^2$/s)')
ax.set_title(f'Enhanced CNT Diffusivity vs Temperature\n'
             rf'$d_{{\mathrm{{CNT}}}}$ = {d_cnt} nm, $M$ = {M_gas} g/mol (N$_2$)')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3, which='both')
ax.set_ylim(1e-8, 1e-2)

plt.tight_layout()
plt.show()

In [ ]:
# Enhancement factor vs specularity parameter
sigma_range = np.linspace(0, 0.995, 200)
F_range = specularity_to_F(sigma_range)

fig, ax = plt.subplots(figsize=(10, 7))

ax.semilogy(sigma_range, F_range, color=COLORS['blue'], linewidth=2.5)

# Mark representative values
sigma_marks = [0, 0.5, 0.9, 0.95, 0.99]
labels_marks = ['Rough pore\n(oxide)', 'Moderate', 'Pristine CNT',
                'High quality\nSWCNT', 'Near-perfect\nspecular']
for sigma, label in zip(sigma_marks, labels_marks):
    F_val = specularity_to_F(sigma)
    ax.plot(sigma, F_val, 'o', color=COLORS['vermillion'], markersize=10,
            markeredgecolor='black', markeredgewidth=1.5, zorder=5)
    ax.annotate(f'{label}\n$F$ = {F_val:.0f}',
                xy=(sigma, F_val),
                xytext=(sigma - 0.1, F_val * 2.5),
                fontsize=9, ha='center',
                arrowprops=dict(arrowstyle='->', color='gray'))

ax.set_xlabel(r'Specularity Parameter, $\sigma$')
ax.set_ylabel('Enhancement Factor, $F$')
ax.set_title(r'Specularity to Enhancement Factor: $F = (1+\sigma)/(1-\sigma)$')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(0.8, 500)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

### Key Observations

1. **Knudsen diffusivity** scales as $D_K \propto d \sqrt{T/M}$ -- proportional to pore diameter and square root of temperature
2. **Enhancement factor** grows dramatically as $\sigma \to 1$: going from 0.9 to 0.99 increases $F$ from 19 to 199
3. At $F = 1000$, CNT diffusivity approaches or exceeds bulk gas diffusivity
4. **Practical implication**: CNT quality (defect density) controls $\sigma$ and therefore $F$. Acid-treated CNTs have lower $F$ than pristine tubes.

**Concept Check:** If the enhancement factor $F = 1000$, by what factor does the critical Thiele modulus (where $\eta$ drops below 0.95) increase compared to classical Knudsen diffusion? Think about the relationship $\phi \propto 1/\sqrt{D_\text{eff}}$.

---
## Part 11: Effectiveness Factor with Enhanced Transport

The enhanced diffusion in CNTs has profound implications for the effectiveness factor.
Replacing $D_{\text{eff}}$ with $F \cdot D_K$ in the Thiele modulus:

$$\phi_{\text{CNT}} = L\sqrt{\frac{k}{F \cdot D_K}} = \frac{\phi_{\text{classical}}}{\sqrt{F}}$$

This means the $\eta$ vs $\phi$ curve effectively **shifts to the right** by $\sqrt{F}$
on a log scale, dramatically expanding the kinetic-control window.

In [ ]:
# effectiveness_slab and effectiveness_sphere are already defined in Part 5.
# Here we add effectiveness_cnt which wraps them with enhancement factor F.

def effectiveness_cnt(phi_classical, F=1, geometry='slab'):
    """
    Calculate effectiveness factor for CNT-enhanced diffusion.

    The classical Thiele modulus is rescaled by 1/sqrt(F) to account
    for enhanced transport: phi_eff = phi_classical / sqrt(F).

    Parameters
    ----------
    phi_classical : float or array-like
        Thiele modulus based on classical Knudsen diffusivity
    F : float
        Enhancement factor (1 = classical, >1 = CNT-enhanced)
    geometry : str, optional
        'slab' or 'sphere'. Default 'slab'.

    Returns
    -------
    float or ndarray
        Effectiveness factor

    Notes
    -----
    phi_CNT = phi_classical / sqrt(F)
    Increasing F shifts the kinetic-to-diffusion transition to higher
    phi_classical values on a log-log plot.
    """
    phi_eff = np.asarray(phi_classical, dtype=float) / np.sqrt(F)
    if geometry == 'slab':
        return effectiveness_slab(phi_eff)
    elif geometry == 'sphere':
        return effectiveness_sphere(phi_eff)
    else:
        raise ValueError(f"Unknown geometry: {geometry}")

In [ ]:
# Effectiveness factor vs classical Thiele modulus for different F
phi_classical = np.logspace(-2, 3, 500)

F_values = [1, 10, 100, 1000]
colors_F = [COLORS['blue'], COLORS['orange'], COLORS['green'],
            COLORS['vermillion']]
linestyles_F = ['-', '--', '-.', ':']

fig, ax = plt.subplots(figsize=(10, 7))

for F, color, ls in zip(F_values, colors_F, linestyles_F):
    eta = effectiveness_cnt(phi_classical, F=F, geometry='slab')
    label = f'F = {F}' if F > 1 else 'Classical (F = 1)'
    ax.loglog(phi_classical, eta, color=color, linestyle=ls,
              linewidth=2.5, label=label)

# Mark the kinetic control boundary (eta = 0.95)
ax.axhline(y=0.95, color='gray', linestyle=':', linewidth=1.5, alpha=0.5)
ax.text(0.015, 0.97, r'$\eta$ = 0.95', fontsize=10, color='gray')

# Show the shift: at phi = 10 (classical diffusion-limited),
# F=100 brings eta back near 1
ax.annotate('F=100 rescues this\nfrom diffusion control',
            xy=(10, effectiveness_cnt(np.array([10.0]), F=100)[0]),
            xytext=(30, 0.5),
            fontsize=10, color=COLORS['green'],
            arrowprops=dict(arrowstyle='->', color=COLORS['green'], lw=2))

# Regime labels
ax.text(0.03, 0.6, 'Kinetic\nControl', fontsize=12, ha='center',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax.text(300, 0.006, 'Diffusion\nControl', fontsize=12, ha='center',
        bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.3))

ax.set_xlabel(r'Classical Thiele Modulus, $\phi_{\mathrm{classical}}$')
ax.set_ylabel(r'Effectiveness Factor, $\eta$')
ax.set_title('Effect of Enhancement Factor on Effectiveness\n'
             r'(Slab geometry, $\phi_{\mathrm{CNT}} = \phi_{\mathrm{classical}} / \sqrt{F}$)')
ax.legend(loc='lower left', fontsize=11)
ax.set_xlim(0.01, 1000)
ax.set_ylim(0.001, 1.5)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

In [ ]:
# Quantify the kinetic control window expansion
# Find the maximum phi_classical that gives eta > 0.95 for each F

print("Kinetic Control Window (eta > 0.95)")
print("=" * 50)
print(f"{'F':>8} {'Max phi_classical':>20} {'Expansion vs F=1':>20}")
print("-" * 50)

phi_test = np.logspace(-2, 4, 10000)
phi_max_F1 = None

for F in [1, 10, 100, 1000, 10000]:
    eta_test = effectiveness_cnt(phi_test, F=F, geometry='slab')
    # Find maximum phi where eta > 0.95
    mask = eta_test > 0.95
    if np.any(mask):
        phi_max = phi_test[mask][-1]
    else:
        phi_max = 0

    if F == 1:
        phi_max_F1 = phi_max

    expansion = phi_max / phi_max_F1 if phi_max_F1 > 0 else 0
    print(f"{F:>8} {phi_max:>20.1f} {expansion:>20.0f}x")

print()
print("Interpretation: F=100 allows 10x higher intrinsic rate constant")
print("while maintaining kinetic control (same as reducing L by 10x).")

### Key Observations

1. **Enhancement factor $F$ shifts the $\eta$ curve to the right by $\sqrt{F}$** on a log scale
2. At $F = 100$: the kinetic-control window expands by a factor of 10 in terms of maximum allowed $\phi$
3. At $F = 1000$: expansion by ~31x -- equivalent to shrinking the catalyst length by 31x
4. **Practical significance**: Enhanced transport allows use of longer CNTs or faster reactions without diffusion limitations

---
## Part 12: CNT Catalyst Design

Given intrinsic kinetics, we can select the CNT tube diameter, length, and required
enhancement factor to achieve a target effectiveness factor. This is the reverse
design problem.

In [ ]:
def design_cnt_catalyst(k_intrinsic, target_eta, L_um, d_cnt_nm,
                        T, M_gmol, geometry='slab'):
    """
    Design a CNT catalyst support by determining required enhancement factor.

    Given intrinsic kinetics and geometric constraints, calculate the
    minimum enhancement factor F needed to achieve the target effectiveness.

    Parameters
    ----------
    k_intrinsic : float
        Intrinsic first-order rate constant (s^-1)
    target_eta : float
        Target effectiveness factor (e.g., 0.95)
    L_um : float
        CNT half-length in micrometers (characteristic length for slab)
    d_cnt_nm : float
        CNT inner diameter (nm)
    T : float
        Temperature (K)
    M_gmol : float
        Molar mass of diffusing species (g/mol)
    geometry : str, optional
        'slab' or 'sphere'. Default 'slab'.

    Returns
    -------
    dict
        Dictionary with design results:
        - 'D_K': Classical Knudsen diffusivity (m^2/s)
        - 'phi_classical': Thiele modulus without enhancement
        - 'eta_classical': Effectiveness factor without enhancement
        - 'F_required': Minimum F needed for target_eta
        - 'phi_enhanced': Thiele modulus with enhancement
        - 'eta_enhanced': Achieved effectiveness factor
        - 'sigma_required': Specularity parameter for required F
    """
    L_m = L_um * 1e-6  # Convert to meters

    # Classical Knudsen diffusivity
    D_K = knudsen_diffusivity(d_cnt_nm, T, M_gmol)

    # Classical Thiele modulus
    phi_classical = L_m * np.sqrt(k_intrinsic / D_K)

    # Classical effectiveness factor
    if geometry == 'slab':
        eta_classical = effectiveness_slab(np.array([phi_classical]))[0]
    else:
        eta_classical = effectiveness_sphere(np.array([phi_classical]))[0]

    # Find required F by searching
    # For slab: eta = tanh(phi/sqrt(F)) / (phi/sqrt(F)) = target_eta
    # Solve numerically by scanning F values
    F_scan = np.logspace(0, 5, 10000)
    eta_scan = effectiveness_cnt(np.array([phi_classical]), F=F_scan,
                                  geometry=geometry)

    # Find minimum F where eta >= target
    # eta_scan has shape (len(F_scan),) because phi_classical is scalar broadcast
    eta_flat = eta_scan.flatten() if eta_scan.ndim > 1 else eta_scan

    # Compute eta for each F value individually
    eta_list = []
    for F_val in F_scan:
        phi_eff = phi_classical / np.sqrt(F_val)
        if geometry == 'slab':
            e = effectiveness_slab(np.array([phi_eff]))[0]
        else:
            e = effectiveness_sphere(np.array([phi_eff]))[0]
        eta_list.append(e)
    eta_array = np.array(eta_list)

    mask = eta_array >= target_eta
    if np.any(mask):
        F_required = F_scan[mask][0]
    elif eta_classical >= target_eta:
        F_required = 1.0
    else:
        F_required = np.inf  # Cannot achieve target

    # Calculate enhanced values
    phi_enhanced = phi_classical / np.sqrt(F_required)
    if geometry == 'slab':
        eta_enhanced = effectiveness_slab(np.array([phi_enhanced]))[0]
    else:
        eta_enhanced = effectiveness_sphere(np.array([phi_enhanced]))[0]

    # Required specularity
    # F = (1+sigma)/(1-sigma)  =>  sigma = (F-1)/(F+1)
    sigma_required = (F_required - 1) / (F_required + 1)

    return {
        'D_K': D_K,
        'phi_classical': phi_classical,
        'eta_classical': eta_classical,
        'F_required': F_required,
        'phi_enhanced': phi_enhanced,
        'eta_enhanced': eta_enhanced,
        'sigma_required': sigma_required
    }

In [ ]:
# Design example: Pt/CNT catalyst for H2 reaction
design = design_cnt_catalyst(
    k_intrinsic=500,    # s^-1 (fast first-order reaction)
    target_eta=0.95,    # Target: >95% effectiveness
    L_um=10,            # CNT half-length: 10 um
    d_cnt_nm=5,         # Inner diameter: 5 nm
    T=400,              # Temperature: 400 K
    M_gmol=2.0,         # H2 molar mass
    geometry='slab'
)

print("CNT Catalyst Design Results")
print("=" * 55)
print(f"Input Parameters:")
print(f"  k = 500 s^-1, L = 10 um, d = 5 nm, T = 400 K")
print(f"  Target: eta > 0.95")
print()
print(f"Classical Knudsen:")
print(f"  D_K = {design['D_K']:.2e} m^2/s")
print(f"  phi = {design['phi_classical']:.2f}")
print(f"  eta = {design['eta_classical']:.4f}")
print()
print(f"Required Enhancement:")
print(f"  F_min = {design['F_required']:.1f}")
print(f"  sigma_min = {design['sigma_required']:.3f}")
print(f"  phi_enhanced = {design['phi_enhanced']:.3f}")
print(f"  eta_enhanced = {design['eta_enhanced']:.4f}")
print()

if design['sigma_required'] < 0.95:
    print("Feasibility: ACHIEVABLE with typical pristine CNTs")
elif design['sigma_required'] < 0.99:
    print("Feasibility: CHALLENGING, requires high-quality CNTs")
else:
    print("Feasibility: DIFFICULT, consider shorter tubes or wider diameter")

In [ ]:
# Design space exploration: F_required vs (k, L) for fixed d and T
k_range = np.logspace(0, 4, 100)  # s^-1
L_range = np.logspace(-1, 2, 100)  # um

K_grid, L_grid = np.meshgrid(k_range, L_range)

# For each (k, L), compute the classical phi
d_design = 5.0  # nm
T_design = 400  # K
M_design = 28.0  # g/mol (N2)
D_K_design = knudsen_diffusivity(d_design, T_design, M_design)

phi_grid = (L_grid * 1e-6) * np.sqrt(K_grid / D_K_design)

# For eta > 0.95 with slab geometry:
# tanh(phi/sqrt(F)) / (phi/sqrt(F)) > 0.95
# phi/sqrt(F) < ~0.35 (numerical inversion)
phi_threshold = 0.35  # phi_eff for eta = 0.95 (slab geometry)

F_required_grid = (phi_grid / phi_threshold)**2
F_required_grid = np.clip(F_required_grid, 1, 1e5)

fig, ax = plt.subplots(figsize=(10, 8))

# Contour plot
levels = [1, 3, 10, 30, 100, 300, 1000, 3000, 10000]
c = ax.contourf(k_range, L_range, F_required_grid,
                levels=levels, cmap='YlOrRd', norm=plt.matplotlib.colors.LogNorm())
cbar = fig.colorbar(c, ax=ax, label='Required Enhancement Factor $F$')

# Contour lines
CS = ax.contour(k_range, L_range, F_required_grid,
                levels=[1, 10, 100, 1000], colors='black',
                linewidths=1.5)
ax.clabel(CS, inline=True, fontsize=10, fmt='F=%g')

# Mark the design point
ax.plot(500, 10, '*', color=COLORS['blue'], markersize=20,
        markeredgecolor='black', markeredgewidth=2, zorder=5,
        label='Design example')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Intrinsic Rate Constant, $k$ (s$^{-1}$)')
ax.set_ylabel(r'CNT Half-Length, $L$ ($\mu$m)')
ax.set_title(f'Design Map: Required $F$ for $\\eta > 0.95$\n'
             rf'$d_{{\mathrm{{CNT}}}}$ = {d_design} nm, T = {T_design} K, '
             f'M = {M_design} g/mol')
ax.legend(loc='upper left')

plt.tight_layout()
plt.show()

---
## Part 13: Comparison -- CNT vs Conventional vs Zeolite Support

We now compare three catalyst support types on the same $\eta$ vs $\phi$ plot
to show where each support type is optimal.

| Support | $D_{\text{eff}}$ (m$^2$/s) | Enhancement | Typical pore/crystal size |
|---------|-----------|-------------|---------------------------|
| Zeolite crystal | $10^{-12}$ to $10^{-10}$ | None (configurational) | 0.1--1 um |
| Conventional (meso) | $10^{-7}$ to $10^{-6}$ | None (Knudsen) | mm pellet |
| CNT (F=100) | $10^{-5}$ to $10^{-4}$ | 100x Knudsen | 1--100 um |

In [ ]:
# Three support types compared on the same effectiveness plot
# For each support, we compute eta vs intrinsic k at fixed geometry

k_range_comp = np.logspace(-1, 5, 500)  # s^-1

# Support parameters
supports = {
    'Zeolite (0.5 um crystal)': {
        'D_eff': 5e-12,       # m^2/s (configurational diffusion)
        'L': 0.5e-6,          # m (crystal radius)
        'color': COLORS['vermillion'],
        'ls': '-',
        'geometry': 'sphere'
    },
    'Mesoporous pellet (1 mm)': {
        'D_eff': 1e-7,        # m^2/s (Knudsen in mesopores)
        'L': 1e-3,            # m (pellet radius)
        'color': COLORS['orange'],
        'ls': '--',
        'geometry': 'sphere'
    },
    'CNT (F=100, 10 um tube)': {
        'D_eff': 1e-7 * 100,  # m^2/s (enhanced Knudsen)
        'L': 10e-6,           # m (half tube length)
        'color': COLORS['blue'],
        'ls': '-.',
        'geometry': 'slab'
    },
    'CNT (F=1000, 10 um tube)': {
        'D_eff': 1e-7 * 1000, # m^2/s (highly enhanced)
        'L': 10e-6,           # m
        'color': COLORS['green'],
        'ls': ':',
        'geometry': 'slab'
    }
}

fig, ax = plt.subplots(figsize=(10, 7))

for name, params in supports.items():
    phi = params['L'] * np.sqrt(k_range_comp / params['D_eff'])
    if params['geometry'] == 'sphere':
        eta = effectiveness_sphere(phi)
    else:
        eta = effectiveness_slab(phi)

    ax.semilogx(k_range_comp, eta, color=params['color'],
                linestyle=params['ls'], linewidth=2.5, label=name)

# Reference lines
ax.axhline(y=0.95, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax.text(0.15, 0.96, r'$\eta$ = 0.95', fontsize=9, color='gray')
ax.axhline(y=0.5, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax.text(0.15, 0.51, r'$\eta$ = 0.50', fontsize=9, color='gray')

ax.set_xlabel('Intrinsic Rate Constant, $k$ (s$^{-1}$)')
ax.set_ylabel(r'Effectiveness Factor, $\eta$')
ax.set_title('Support Comparison: Effectiveness Factor vs. Intrinsic Rate')
ax.set_ylim(0, 1.05)
ax.legend(loc='lower left', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print the maximum k for eta > 0.95 for each support
print("\nMaximum intrinsic k for eta > 0.95:")
print("=" * 55)
for name, params in supports.items():
    phi = params['L'] * np.sqrt(k_range_comp / params['D_eff'])
    if params['geometry'] == 'sphere':
        eta = effectiveness_sphere(phi)
    else:
        eta = effectiveness_slab(phi)
    mask = eta > 0.95
    if np.any(mask):
        k_max = k_range_comp[mask][-1]
        print(f"  {name:<35} k_max = {k_max:.1f} s^-1")
    else:
        print(f"  {name:<35} Always below 0.95")

### Key Observations

1. **Zeolite crystals** become diffusion-limited at very low $k$ due to extremely slow configurational diffusion -- compensated by sub-micron crystal sizes
2. **Mesoporous pellets** tolerate moderate $k$ values before diffusion limitation sets in
3. **CNT supports** dramatically extend the kinetic-control window thanks to enhanced transport
4. **Optimal support selection** depends on the intrinsic rate constant:
   - Slow reactions ($k < 1$ s$^{-1}$): Any support works
   - Moderate reactions: Mesoporous or CNT
   - Fast reactions ($k > 100$ s$^{-1}$): CNT with high $F$ is essential

---
## Part 14: Interactive Parameter Exploration

Modify the parameters below to explore different CNT catalyst designs.

In [ ]:
# =========================================================================
# ADJUSTABLE PARAMETERS - Modify these to explore different scenarios
# =========================================================================

# CNT properties
n_chiral = 10              # Chiral index n
m_chiral = 0               # Chiral index m
F_enhancement = 100        # Enhancement factor
L_tube_um = 5.0            # Tube half-length (um)

# Reaction conditions
T_reaction = 400           # Temperature (K)
k_reaction = 200           # Intrinsic rate constant (s^-1)
M_reactant = 28.0          # Molar mass (g/mol) -- N2

# =========================================================================
# CALCULATIONS (do not modify)
# =========================================================================

# CNT structure
cnt_info = cnt_diameter(n_chiral, m_chiral)
d_inner = cnt_info['diameter_nm']

# Diffusivities
D_K_val = knudsen_diffusivity(d_inner, T_reaction, M_reactant)
D_CNT_val = D_K_val * F_enhancement

# Thiele modulus and effectiveness
L_m = L_tube_um * 1e-6
phi_class = L_m * np.sqrt(k_reaction / D_K_val)
phi_enh = L_m * np.sqrt(k_reaction / D_CNT_val)
eta_class = effectiveness_slab(np.array([phi_class]))[0]
eta_enh = effectiveness_slab(np.array([phi_enh]))[0]

print("CNT Catalyst Performance Summary")
print("=" * 55)
print(f"CNT Structure: ({n_chiral},{m_chiral})")
print(f"  Diameter:      {d_inner:.3f} nm")
print(f"  Chirality:     {cnt_info['chirality']}")
print(f"  Electronic:    {cnt_info['electronic']}")
print(f"  Tube length:   {L_tube_um:.1f} um (half-length)")
print()
print(f"Transport:")
print(f"  D_K (classical):  {D_K_val:.2e} m^2/s")
print(f"  D_CNT (F={F_enhancement}):  {D_CNT_val:.2e} m^2/s")
print()
print(f"Performance (k = {k_reaction} s^-1):")
print(f"  Without enhancement: phi = {phi_class:.2f}, eta = {eta_class:.4f}")
print(f"  With enhancement:    phi = {phi_enh:.3f}, eta = {eta_enh:.4f}")
print(f"  Activity gain from F: {eta_enh/eta_class:.1f}x")
print()

# Maximum nanoparticle diameter that fits inside
d_np_max = d_inner * 0.8  # Rule of thumb: NP < 80% of tube diameter
print(f"Max nanoparticle diameter (80% rule): {d_np_max:.2f} nm")

---
## Exercises

### Exercise 1: Acid Site Density for ZSM-5

An H-ZSM-5 (MFI framework) has Si/Al = 25. The MFI unit cell contains 96 T-atoms and 192 oxygen atoms.

**Tasks:**
1. Calculate the theoretical acid site density in mmol/g
2. If NH3-TPD measures 0.35 mmol/g, what fraction of Al generates accessible Brønsted sites?
3. Plot acid site density vs Si/Al ratio from 5 to 100 for the MFI framework

In [ ]:
# YOUR CODE HERE
# 1. Use the acid_site_density function with Si/Al = 25, T_atoms = 96
# 2. Compute fraction = measured / theoretical
# 3. Plot for Si/Al = 5 to 100

pass  # Replace with your implementation

### Exercise 2: Maximum Molecule Size for CHA Pores

The CHA framework (SSZ-13) has 8-MR windows with a diameter of 3.8 angstrom.

**Tasks:**
1. Using the molecular diameter table from Part 3, determine which molecules can enter CHA pores
2. Explain why CHA-based Cu-SSZ-13 is effective for NH3-SCR (NH3 kinetic diameter ~2.6 angstrom, NO ~3.2 angstrom) but not for large-molecule reactions
3. What is the maximum kinetic diameter of a molecule that could enter CHA with a 0.3 angstrom tolerance?

In [ ]:
# YOUR CODE HERE
# 1. Filter molecules from the table that can enter CHA pores
# 2. Explain the SCR application in a print statement
# 3. Calculate max diameter = 3.8 + tolerance

pass  # Replace with your implementation

### Exercise 3: Zeolite Catalyst Design for n-Hexane Cracking

Design an H-Y (FAU) zeolite catalyst for n-hexane cracking with the following specifications:
- Intrinsic rate constant: $k = 50$ s$^{-1}$ at 500 degrees C
- Target: effectiveness factor $\eta > 0.8$

**Tasks:**
1. For n-hexane in FAU, estimate $D_{\text{micro}} \approx 1 \times 10^{-10}$ m$^2$/s (larger pores = faster diffusion)
2. Calculate the maximum crystal radius that achieves $\eta > 0.8$
3. Plot $\eta$ vs crystal radius and identify the design window
4. Compare with the same reaction in MFI where $D_{\text{micro}} \approx 5 \times 10^{-12}$ m$^2$/s

In [ ]:
# YOUR CODE HERE
# 1. Set D_micro for FAU
# 2. Find phi that gives eta = 0.8 for sphere geometry,
#    then R_max = phi * sqrt(D_micro / k)
# 3. Plot eta vs R
# 4. Repeat for MFI

pass  # Replace with your implementation

### Exercise 4: Maximum Nanoparticle Diameter for (10,0) Tube

For a (10,0) zigzag SWCNT:

**Tasks:**
1. Calculate the tube diameter using `cnt_diameter(10, 0)`
2. Determine the electronic character (metallic or semiconducting?)
3. What is the maximum metal nanoparticle diameter that could fit inside? (Assume the NP must be smaller than 80% of the tube inner diameter to allow gas transport)
4. Is this tube suitable for confining Pt nanoparticles (typical 2--5 nm)?

In [ ]:
# YOUR CODE HERE
# 1. Call cnt_diameter(10, 0)
# 2. Check the 'electronic' key
# 3. Calculate 0.8 * diameter
# 4. Compare with typical Pt NP sizes

pass  # Replace with your implementation

### Exercise 5: Enhancement Factor for Kinetic Control

A CNT-supported catalyst has the following specifications:
- CNT half-length: $L = 10$ um
- CNT inner diameter: $d = 5$ nm
- Intrinsic rate constant: $k = 500$ s$^{-1}$
- Temperature: $T = 400$ K
- Gas: N$_2$ ($M = 28$ g/mol)

**Tasks:**
1. Calculate the classical Knudsen diffusivity
2. Calculate the classical Thiele modulus
3. Determine $\eta$ without enhancement
4. Find the minimum $F$ needed for $\eta > 0.95$
5. What specularity parameter $\sigma$ does this correspond to? Is it achievable?

In [ ]:
# YOUR CODE HERE
# 1. Use knudsen_diffusivity(5, 400, 28)
# 2. phi = L * sqrt(k / D_K)
# 3. effectiveness_slab(phi)
# 4. Use design_cnt_catalyst or scan F values
# 5. sigma = (F-1)/(F+1)

pass  # Replace with your implementation

### Exercise 6: Slab vs Sphere Geometry for CNT Effectiveness

Compare the effectiveness factor using slab and sphere geometry for a CNT catalyst.

**Tasks:**
1. Plot $\eta$ vs $\phi_{\text{classical}}$ for both geometries with $F = 100$
2. At what classical $\phi$ does the choice of geometry make a significant difference (>10% difference in $\eta$)?
3. Discuss which geometry is more appropriate for modeling a CNT channel

In [ ]:
# YOUR CODE HERE
# 1. Use effectiveness_cnt with geometry='slab' and geometry='sphere'
# 2. Compute |eta_slab - eta_sphere| / eta_slab and find where > 0.1
# 3. Discuss in a print statement

pass  # Replace with your implementation

---
## Reflection Questions

### Zeolites

1. Hierarchical zeolites contain both micropores (for shape selectivity) and mesopores (for improved transport). How would you expect the effectiveness factor curve to change for a hierarchical zeolite compared to a purely microporous one? Would shape selectivity be affected?

2. The para-xylene selectivity model assumes fast re-equilibration inside the crystal. Under what conditions would this assumption break down, and how would it affect selectivity?

3. Løwenstein's rule sets Si/Al >= 1 as the maximum aluminum content. What would happen to acid site strength if you could somehow violate this rule? Why is it thermodynamically forbidden?

4. A zeolite catalyst shows decreasing activity over time but unchanged selectivity. Based on what you know about zeolite structure, propose a deactivation mechanism consistent with these observations.

### Carbon Nanotubes

5. The enhancement factor $F$ can be as high as 10,000 for pristine CNTs. Under what conditions would you expect $F$ to decrease, and how would this affect catalyst design?

6. CNT-supported catalysts are more expensive than conventional supports. In what types of reactions would the enhanced transport justify the additional cost?

7. The $(n,m)$ map shows that approximately 1/3 of CNTs are metallic. For electrocatalysis applications, would you want metallic or semiconducting CNTs as supports? Why?

### Connecting Both

8. Compare the transport limitations in zeolites with those in CNTs. Both have nanoscale pores, yet their diffusion behaviors are opposite (extremely slow vs. extremely fast). What structural feature explains this difference?

---
## Summary

In this notebook, we have:

### Zeolites (Parts 1--7)

1. **Compared zeolite frameworks** by pore size, channel dimensionality, and cage geometry
2. **Calculated acid site density** from Si/Al ratio and unit cell composition using Løwenstein's rule
3. **Modeled shape selectivity** using molecular size vs. pore size criteria across all three selectivity types
4. **Compared diffusion regimes**: bulk > Knudsen > configurational, with orders-of-magnitude differences
5. **Computed effectiveness factors** for zeolite crystals, showing that crystal size is the critical design variable
6. **Analyzed para-xylene selectivity** as a product shape selectivity case study in MFI zeolite

### Carbon Nanotubes (Parts 8--14)

7. **Implemented CNT structure calculations** from chiral indices, including diameter, chirality, and electronic character
8. **Compared SWCNT and MWCNT** properties relevant to catalytic applications
9. **Computed Knudsen and enhanced diffusivities**, showing that CNT enhancement factors of 10--10,000 arise from specular reflection on atomically smooth walls
10. **Quantified how enhanced transport expands the kinetic-control window** using the $\eta$ vs $\phi$ framework
11. **Designed a CNT catalyst** by selecting tube geometry and required enhancement factor
12. **Compared three support types** (zeolite, mesoporous, CNT) to identify the optimal support for different reaction rate regimes

### Key Contrasts

| Property | Zeolite Micropores | CNT Channels |
|----------|-------------------|---------------|
| Pore diameter | 3--8 Å | 0.4--100 nm |
| Diffusion regime | Configurational | Enhanced Knudsen |
| $D_{\text{eff}}$ (m$^2$/s) | $10^{-10}$ to $10^{-18}$ | $10^{-5}$ to $10^{-3}$ |
| Key advantage | Shape selectivity | Expanded kinetic window |
| Design lever | Crystal size | Enhancement factor $F$ |

### Key Functions Defined

- `acid_site_density(si_al_ratio, T_atoms_per_UC)` -- Brønsted acid sites from Si/Al
- `check_molecular_access(d_molecule, d_pore)` -- Shape selectivity check
- `bulk_diffusivity(T, P, M_A, M_B)` -- Chapman-Enskog estimate
- `effectiveness_sphere(phi)` -- Spherical geometry $\eta(\phi)$
- `effectiveness_slab(phi)` -- Slab geometry $\eta(\phi)$
- `cnt_diameter(n, m)` -- CNT structure from chiral indices
- `knudsen_diffusivity(d_pore_nm, T, M_gmol)` -- Classical Knudsen $D_K$
- `enhanced_diffusivity(d_pore_nm, T, M_gmol, F)` -- CNT-enhanced $D = F \cdot D_K$
- `specularity_to_F(sigma)` -- Specularity parameter to enhancement factor
- `effectiveness_cnt(phi_classical, F, geometry)` -- Enhanced effectiveness factor
- `design_cnt_catalyst(k, target_eta, L, d, T, M)` -- Reverse design problem

**Next:** NB-S (Capstone) applies concepts from the entire course.